# RiskLens — Risk Methodology Notebook

This notebook explains the quantitative risk methods used in RiskLens:
1. Value at Risk (VaR) — worked numerical example
2. Historical vs Parametric VaR — comparison on real data
3. Expected Shortfall (CVaR) — why Basel III moved from VaR to CVaR
4. The three Basel III pillars — plain English
5. Credit Risk Scoring — how PD is estimated
6. Yield Curve and Recession Signals
7. How LangGraph works — simple 3-node example

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.facecolor'] = '#0d0d18'
plt.rcParams['figure.facecolor'] = '#0d0d18'

TRADING_DAYS = 252
print('Setup complete.')

---
## 1. What is Value at Risk (VaR)?

**VaR** answers the question: *"What is the maximum loss I can expect over a given period, with a given level of confidence?"

**Formal definition**: 1-day 95% VaR = the loss threshold such that there is only a 5% probability of losing MORE than this amount on any given day.

**Example**: If your portfolio's 1-day 95% VaR is $1.5M on a $100M portfolio (1.5% of AUM), that means:
- On 95% of trading days, you will NOT lose more than $1.5M
- On 5% of trading days (about 12-13 days per year), you WILL lose more than $1.5M
- VaR says NOTHING about HOW MUCH MORE you lose on those 5% of days

The last point — VaR's silence about the tail — is its critical flaw and why Basel III moved to CVaR.

In [ ]:
# Worked numerical example: VaR on a simple portfolio
np.random.seed(42)

# Simulate 252 days of portfolio log returns (mean=0.03%/day, std=1%/day)
n_days = 252
portfolio_value = 100_000_000  # $100M portfolio
daily_returns = np.random.normal(loc=0.0003, scale=0.010, size=n_days)

# Historical VaR: just read the 5th percentile
var_95_fraction = -np.percentile(daily_returns, 5)
var_99_fraction = -np.percentile(daily_returns, 1)

var_95_dollars = var_95_fraction * portfolio_value
var_99_dollars = var_99_fraction * portfolio_value

print(f'Portfolio: ${portfolio_value:,.0f}')
print(f'Return series: {n_days} trading days')
print(f'Mean daily return: {daily_returns.mean()*100:.3f}%')
print(f'Daily volatility: {daily_returns.std()*100:.3f}%')
print()
print(f'1-day 95% VaR: {var_95_fraction*100:.3f}% = ${var_95_dollars:,.0f}')
print(f'1-day 99% VaR: {var_99_fraction*100:.3f}% = ${var_99_dollars:,.0f}')
print()
print(f'Interpretation: On 95% of days, daily loss will NOT exceed ${var_95_dollars:,.0f}')
print(f'Expected VaR exceptions per year: {n_days * 0.05:.1f} days (5% of 252)')

In [ ]:
# Visualise the return distribution with VaR thresholds
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(daily_returns * 100, bins=40, color='#3355bb', alpha=0.7, label='Daily Returns')
ax.axvline(-var_95_fraction * 100, color='#eab308', linestyle='--', linewidth=2, label=f'95% VaR = {var_95_fraction*100:.2f}%')
ax.axvline(-var_99_fraction * 100, color='#dc2626', linestyle='--', linewidth=2, label=f'99% VaR = {var_99_fraction*100:.2f}%')

# Shade the tail regions
tail_returns = daily_returns[daily_returns < -var_95_fraction]
ax.hist(tail_returns * 100, bins=20, color='#dc2626', alpha=0.5, label='5% Tail (VaR exceptions)')

ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
ax.set_title('Portfolio Return Distribution with VaR Thresholds')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f'\nNumber of days beyond 95% VaR threshold: {(daily_returns < -var_95_fraction).sum()}')
print(f'Expected (5% × 252): {252 * 0.05:.0f}')

---
## 2. Historical vs Parametric VaR

**Historical Simulation**: Uses the actual distribution of observed returns. No assumption about the return distribution. VaR = the (1-confidence)th percentile of the historical return distribution.

**Parametric (Gaussian) VaR**: Assumes returns are normally distributed. Uses mean and standard deviation only.
```
Parametric VaR = -(μ + z × σ)
```
Where z is the standard normal quantile (z = -1.645 for 95%, z = -2.326 for 99%).

**Key difference**: In calm periods, both methods agree. In volatile periods or for fat-tailed distributions, historical simulation gives higher (more conservative) VaR because it captures the actual tail shape.

In [ ]:
# Compare both methods on the same return series
import yfinance as yf

# Download real SPY data for comparison
try:
    spy_data = yf.download('SPY', period='3y', auto_adjust=True, progress=False)
    spy_returns = np.log(spy_data['Close'] / spy_data['Close'].shift(1)).dropna().values
    print(f'Downloaded {len(spy_returns)} days of SPY returns')
except Exception as e:
    print(f'yfinance failed ({e}) — using simulated data')
    spy_returns = np.random.normal(0.0004, 0.012, 756)  # 3 years simulated

# Historical VaR
hist_var95 = -np.percentile(spy_returns, 5)
hist_var99 = -np.percentile(spy_returns, 1)

# Parametric VaR
mu, sigma = spy_returns.mean(), spy_returns.std()
z95 = stats.norm.ppf(0.05)  # -1.645
z99 = stats.norm.ppf(0.01)  # -2.326
param_var95 = -(mu + z95 * sigma)
param_var99 = -(mu + z99 * sigma)

# Kurtosis (fat-tail measure)
kurtosis = stats.kurtosis(spy_returns)

print(f'\n--- Return Distribution Statistics ---')
print(f'Mean (daily):    {mu*100:.4f}%  | Annualised: {mu*252*100:.2f}%')
print(f'Std Dev (daily): {sigma*100:.4f}% | Annualised: {sigma*np.sqrt(252)*100:.2f}%')
print(f'Excess Kurtosis: {kurtosis:.2f} (normal=0; positive = fat tails)')
print()
print(f'--- VaR Comparison ---')
print(f'              Historical    Parametric    Difference')
print(f'95% VaR:      {hist_var95*100:.3f}%        {param_var95*100:.3f}%         {(hist_var95-param_var95)*100:+.3f}%')
print(f'99% VaR:      {hist_var99*100:.3f}%        {param_var99*100:.3f}%         {(hist_var99-param_var99)*100:+.3f}%')
print()
print(f'Key insight: If kurtosis > 0, historical VaR > parametric VaR at 99%')
print(f'because fat tails mean more extreme losses than a normal distribution predicts.')

In [ ]:
# Visualise: actual return distribution vs normal fit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution overlay
ax = axes[0]
ax.hist(spy_returns * 100, bins=60, density=True, color='#3355bb', alpha=0.6, label='Actual Returns')
x = np.linspace(spy_returns.min() * 100, spy_returns.max() * 100, 300)
ax.plot(x, stats.norm.pdf(x, mu * 100, sigma * 100), 'r-', linewidth=2, label='Normal Fit')
ax.axvline(-hist_var99 * 100, color='#eab308', linestyle='--', linewidth=2, label=f'Hist 99% VaR')
ax.axvline(-param_var99 * 100, color='#dc2626', linestyle='--', linewidth=2, label=f'Param 99% VaR')
ax.set_xlim(-6, 6)
ax.set_xlabel('Daily Return (%)')
ax.set_title('Return Distribution: Actual vs Normal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Right: Q-Q plot (shows fat tails clearly)
ax2 = axes[1]
stats.probplot(spy_returns, dist='norm', plot=ax2)
ax2.get_lines()[0].set(color='#3355bb', alpha=0.5, markersize=2)
ax2.get_lines()[1].set(color='#dc2626', linewidth=2)
ax2.set_title('Q-Q Plot (deviations from line = fat tails)')
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()
print('In the Q-Q plot: if returns were normal, all points would fall on the red line.')
print('The S-curve shape shows fat tails — extreme returns are more common than normal predicts.')

---
## 3. Expected Shortfall (CVaR) — Why Basel III Moved Away from VaR

**VaR's critical flaw**: It tells you the loss THRESHOLD but nothing about the losses BEYOND that threshold. Two portfolios can have identical VaR but completely different risk profiles if one has a thin tail and the other has a fat tail.

**Expected Shortfall (ES / CVaR)** fixes this by measuring the **average loss in the worst (1-confidence)% of scenarios**:
```
CVaR(95%) = E[Loss | Loss > VaR(95%)]
           = Average of all losses in the worst 5% of days
```

**Basel IV / FRTB (2019)**: Replaced 99% VaR with 97.5% ES as the primary market risk metric. The reason:
1. ES is **sub-additive**: ES(A+B) ≤ ES(A) + ES(B) — diversification always reduces risk
2. ES captures the **shape of the tail**, not just its starting point
3. ES is more **consistent** — less sensitive to how you define the return window

In [ ]:
# CVaR calculation and comparison with VaR
confidence = 0.95
var_threshold = hist_var95  # already computed above

# CVaR = average of all losses worse than VaR
tail_losses = spy_returns[spy_returns < -var_threshold]
cvar_95 = -tail_losses.mean()

print(f'--- VaR vs CVaR Comparison (SPY) ---')
print(f'95% VaR:   {hist_var95*100:.3f}%  (loss on the worst 5% of days: minimum)')
print(f'95% CVaR:  {cvar_95*100:.3f}%  (loss on the worst 5% of days: average)')
print(f'CVaR/VaR:  {cvar_95/hist_var95:.2f}x  (CVaR is this much larger than VaR)')
print()
print(f'Number of tail observations: {len(tail_losses)}')
print(f'Worst single day return: {spy_returns.min()*100:.3f}%')
print()
print(f'Insight: CVaR tells you that on the {len(tail_losses)} worst days,')
print(f'the average loss was {cvar_95*100:.3f}%, not just the minimum {hist_var95*100:.3f}%.')
print(f'This is what a risk manager needs to know for capital allocation.')

In [ ]:
# Illustrate why two portfolios can have same VaR but different CVaR
np.random.seed(42)

# Portfolio A: normal-ish tails
port_a = np.concatenate([
    np.random.normal(0.0005, 0.008, 240),  # 240 normal days
    np.random.uniform(-0.020, -0.015, 12)   # 12 mild tail days
])

# Portfolio B: fat tail — same VaR, but much worse CVaR
port_b = np.concatenate([
    np.random.normal(0.0005, 0.008, 240),  # 240 normal days
    np.random.uniform(-0.060, -0.020, 12)  # 12 severe tail days
])

var_a = -np.percentile(port_a, 5)
var_b = -np.percentile(port_b, 5)
cvar_a = -port_a[port_a < -var_a].mean()
cvar_b = -port_b[port_b < -var_b].mean()

print(f'             Portfolio A    Portfolio B')
print(f'95% VaR:     {var_a*100:.3f}%         {var_b*100:.3f}%')
print(f'95% CVaR:    {cvar_a*100:.3f}%         {cvar_b*100:.3f}%')
print()
print('Both portfolios have similar VaR — but B has 2x higher CVaR.')
print('VaR would treat them identically. CVaR correctly identifies B as riskier.')
print('This is the core argument for why Basel III moved to CVaR.')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for i, (port, name, color) in enumerate([(port_a, 'Portfolio A (thin tail)', '#22c55e'), (port_b, 'Portfolio B (fat tail)', '#dc2626')]):
    ax = axes[i]
    var_val = -np.percentile(port, 5)
    cvar_val = -port[port < -var_val].mean()
    ax.hist(port * 100, bins=30, color=color, alpha=0.6)
    ax.axvline(-var_val * 100, color='#eab308', linestyle='--', linewidth=2, label=f'VaR 95%: {var_val*100:.2f}%')
    ax.axvline(-cvar_val * 100, color='white', linestyle='--', linewidth=2, label=f'CVaR 95%: {cvar_val*100:.2f}%')
    ax.set_title(name)
    ax.set_xlabel('Daily Return (%)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 4. The Three Basel III Pillars — Plain English

Basel III is built on three pillars that work together:

### Pillar 1: Minimum Capital Requirements
**"How much capital must you hold?"**

Banks must hold capital proportional to their Risk-Weighted Assets (RWA):
- **CET1 ratio ≥ 7.0%** (4.5% minimum + 2.5% conservation buffer)
- **Tier 1 ratio ≥ 8.5%**
- **Total Capital ≥ 10.5%**

The three types of risk that generate RWA:
- **Credit Risk** (~75-80% of RWA): loans, bonds, counterparty exposures
- **Market Risk** (~10-15% of RWA): trading book positions
- **Operational Risk** (~10-15% of RWA): fraud, systems failures, legal

### Pillar 2: Supervisory Review
**"Are your internal models actually good?"**

OSFI (Canada) / ECB / Fed review the bank's ICAAP (Internal Capital Adequacy Assessment Process). They can add Pillar 2 capital add-ons if they think the bank's models understate risk. This is why a bank's actual capital ratio is often higher than the regulatory minimum — management buffers plus Pillar 2 requirements.

### Pillar 3: Market Discipline
**"Show your work to the public."**

Mandatory disclosures of capital ratios, RWA breakdown, VaR/CVaR figures, and liquidity metrics. The idea: if investors and counterparties can see your risk profile, market pressure reinforces regulatory pressure.

In [ ]:
# Visualise the Basel III capital stack
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['CET1\n(Minimum)', 'CET1\n(+ Buffer)', 'Tier 1', 'Total Capital']
minimums = [4.5, 7.0, 8.5, 10.5]
bars = ax.barh(categories, minimums, color=['#dc2626', '#eab308', '#f97316', '#22c55e'], alpha=0.8)

for bar, val in zip(bars, minimums):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val}%',
            va='center', color='white', fontweight='bold')

ax.set_xlabel('% of Risk-Weighted Assets (RWA)')
ax.set_title('Basel III Minimum Capital Requirements')
ax.set_xlim(0, 14)
ax.axvline(7.0, color='white', linestyle=':', alpha=0.3, label='Effective CET1 minimum (incl. buffer)')
ax.legend()
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

print('Key numbers to know for a bank interview:')
print('  CET1 minimum: 4.5% | + 2.5% conservation buffer = 7.0% effective minimum')
print('  Canadian G-SIBs (RBC, TD, Scotia): +1-2.5% G-SIB surcharge on top')
print('  OSFI Domestic Stability Buffer (Q4 2023): +3.5% = total CET1 target ~10.5%+')

---
## 5. Credit Risk Scoring — How PD is Estimated

**Probability of Default (PD)** is the cornerstone of credit risk measurement under Basel III's IRB approach.

**For public companies**, several proxy methods exist:

### Method 1: Equity Volatility-Implied PD (Merton Model)
- Uses equity market cap + balance sheet debt
- Treats equity as a call option on the firm's assets
- Distance to Default (DD) = how many standard deviations away from insolvency

### Method 2: Altman Z-Score
- Simple linear model using 5 financial ratios
- Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5

### Method 3: CDS Spread Implied Rating
- 5-year CDS spread maps approximately to a credit rating and PD
- CDS < 50bps ≈ investment grade | CDS > 300bps ≈ distressed

In [ ]:
# Simplified Merton distance-to-default illustration
# The concept: equity holders own a call option on firm assets

firm_asset_value = 100   # $100M in assets
debt_face_value = 70     # $70M in debt (leverage = 70%)
asset_volatility = 0.20  # 20% annual vol of firm assets
T = 1.0                  # 1-year horizon

# Distance to Default: how many std devs is current value above the default trigger
# Simplified formula (not the full KMV model)
expected_asset_value = firm_asset_value * np.exp(0.05 * T)  # assume 5% drift
distance_to_default = (np.log(expected_asset_value / debt_face_value)) / (asset_volatility * np.sqrt(T))
pd_estimate = stats.norm.cdf(-distance_to_default)  # Normal CDF of negative DD

print(f'--- Simplified Merton Model ---')
print(f'Firm Asset Value: ${firm_asset_value}M')
print(f'Debt (default trigger): ${debt_face_value}M')
print(f'Asset Volatility: {asset_volatility*100:.0f}% per year')
print(f'Expected Asset Value in 1yr: ${expected_asset_value:.1f}M')
print(f'Distance to Default: {distance_to_default:.2f} standard deviations')
print(f'Estimated PD: {pd_estimate*100:.4f}%')
print()
print('Sensitivity analysis — how does leverage affect PD?')
for leverage in [50, 60, 70, 80, 85, 90]:
    debt = leverage
    dd = (np.log(expected_asset_value / debt)) / (asset_volatility * np.sqrt(T))
    pd = stats.norm.cdf(-dd) * 100
    grade = 'AAA' if pd < 0.01 else ('AA' if pd < 0.05 else ('A' if pd < 0.10 else ('BBB' if pd < 0.25 else ('BB' if pd < 2.0 else 'B+'))))
    print(f'  Leverage {leverage}%: DD={dd:.2f}, PD={pd:.4f}% ({grade})')

---
## 6. Yield Curve and Recession Signals

The **10Y-2Y Treasury yield spread** (FRED series T10Y2Y) is the most reliable leading recession indicator in the US:
- **Normal**: 10Y yield > 2Y yield (positive spread) — investors demand premium for holding longer bonds
- **Flat**: 10Y ≈ 2Y — uncertainty about the economic path
- **Inverted**: 10Y < 2Y — market expects the Fed to cut rates (because of a recession)

**Historical record**: The yield curve inverted before every US recession since 1960. Average lead time: 12-18 months.

- **1989 inversion**: Preceded 1990-91 recession
- **2000 inversion**: Preceded 2001 dot-com recession
- **2006 inversion**: Preceded 2008-09 Great Recession
- **2019 inversion**: Preceded 2020 COVID recession (though COVID was exogenous, the economy was already slowing)
- **2022-2023 inversion**: Deepest inversion since the 1980s — outcome still unfolding

In [ ]:
# Illustrate yield curve inversion signal with simulated data
# (Real FRED data requires API key — using illustrative simulation)

np.random.seed(123)
years = np.arange(2015, 2026, 1/12)  # Monthly data 2015-2025
n = len(years)

# Simulate a realistic yield curve spread scenario
# Normal positive spread → inversion 2019 → recovery → inversion again 2022-2023
def simulate_spread(years):
    spreads = []
    for i, y in enumerate(years):
        if y < 2018.5:
            base = 1.2 - (y - 2015) * 0.2  # Gradually flattening
        elif y < 2020:
            base = 1.2 - (y - 2015) * 0.2 - 0.8 * max(0, y - 2018.5)  # Inversion
        elif y < 2022:
            base = 0.8 + (y - 2020) * 0.8  # Recovery / steepening
        else:
            base = 2.0 - (y - 2022) * 2.0  # Second inversion
        spreads.append(base + np.random.normal(0, 0.15))
    return np.array(spreads)

spread = simulate_spread(years)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(years, spread, '#3355bb', linewidth=1.5, label='10Y-2Y Spread (%)')
ax.fill_between(years, spread, 0, where=(spread < 0), color='#dc2626', alpha=0.4, label='Inverted (recession warning)')
ax.fill_between(years, spread, 0, where=(spread >= 0), color='#22c55e', alpha=0.2, label='Normal')
ax.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)

# Annotate inversions
ax.annotate('2019 Inversion\n(COVID recession 2020)', xy=(2019.2, -0.1), xytext=(2018, -1.2),
            arrowprops=dict(arrowstyle='->', color='orange'), color='orange', fontsize=9)
ax.annotate('2022-23 Inversion\n(Deepest since 1980s)', xy=(2023, -0.8), xytext=(2023.5, -1.8),
            arrowprops=dict(arrowstyle='->', color='orange'), color='orange', fontsize=9)

ax.set_xlabel('Year')
ax.set_ylabel('Yield Spread (%)')
ax.set_title('Simulated 10Y-2Y Treasury Spread (Yield Curve Inversion Indicator)')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

inversion_months = (spread < 0).sum()
print(f'Months in inversion: {inversion_months} ({inversion_months/len(years)*100:.1f}% of the period)')
print(f'Typical lag from inversion to recession onset: 12-18 months')
print(f'Why does inversion cause concern for a bank portfolio?')
print(f'  1. Duration risk: long-term bond prices fall when short rates rise above long rates')
print(f'  2. Net Interest Margin compression: banks borrow short and lend long')
print(f'  3. Credit cycle signal: economic slowdown typically follows, raising default risk')

---
## 7. How LangGraph Works — Simple 3-Node Example

LangGraph is a framework for building stateful, multi-agent workflows. The core concept:

1. **State**: A shared TypedDict that all nodes read from and write to
2. **Nodes**: Functions that transform state (agents, tools, decision logic)
3. **Edges**: Define the flow — fixed edges, conditional edges, or parallel edges (Send API)
4. **Checkpointer**: Persists state between invocations (enables HITL interrupts)

In RiskLens:
- The **Supervisor** is a node that makes routing decisions
- **Specialist agents** are nodes that enrich the state with risk metrics
- The **interrupt()** in the HITL node pauses the graph at that node until `.invoke()` is called again
- The **SQLite checkpointer** makes this pause durable — the state survives process restarts

In [ ]:
# Simple 3-node LangGraph example to illustrate the concepts
# (no LLM calls — pure state manipulation)

from typing import TypedDict, List
from langgraph.graph import StateGraph, END

# 1. Define the state
class SimpleState(TypedDict):
    numbers: List[int]
    running_sum: int
    message: str

# 2. Define nodes (pure functions that transform state)
def node_double(state: SimpleState) -> SimpleState:
    """Double all numbers in the list."""
    doubled = [n * 2 for n in state['numbers']]
    return {**state, 'numbers': doubled, 'message': f'Doubled: {doubled}'}

def node_sum(state: SimpleState) -> SimpleState:
    """Sum the numbers."""
    total = sum(state['numbers'])
    return {**state, 'running_sum': total, 'message': f'Sum: {total}'}

def node_report(state: SimpleState) -> SimpleState:
    """Generate final report."""
    report = f'Processed {len(state["numbers"])} numbers. Sum={state["running_sum"]}'
    return {**state, 'message': report}

# 3. Build the graph
workflow = StateGraph(SimpleState)
workflow.add_node('double', node_double)
workflow.add_node('sum', node_sum)
workflow.add_node('report', node_report)

# 4. Define edges
workflow.set_entry_point('double')
workflow.add_edge('double', 'sum')
workflow.add_edge('sum', 'report')
workflow.add_edge('report', END)

# 5. Compile and run
graph = workflow.compile()

initial_state = SimpleState(numbers=[1, 2, 3, 4, 5], running_sum=0, message='')
result = graph.invoke(initial_state)

print('Simple LangGraph example:')
print(f'Input:  {initial_state["numbers"]}')
print(f'After double: {[n*2 for n in initial_state["numbers"]]}  (node_double)')
print(f'After sum: {sum(n*2 for n in initial_state["numbers"])}  (node_sum)')
print(f'Final message: {result["message"]}  (node_report)')
print()
print('This is the exact same pattern as RiskLens:')
print('  node_double → market_risk_agent (enriches state with VaR, vol, etc.)')
print('  node_sum    → aggregator_node (computes composite risk score)')
print('  node_report → report_agent (generates final briefing)')
print()
print('The power: each node ONLY reads what it needs from state and writes its own fields.')
print('Nodes can run in parallel (Send API) because they write to different state keys.')

In [ ]:
# Visualise the LangGraph architecture with a simple diagram
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

def draw_node(ax, x, y, label, color, width=2.5, height=0.8):
    rect = mpatches.FancyBboxPatch((x - width/2, y - height/2), width, height,
                                    boxstyle='round,pad=0.1', facecolor=color, edgecolor='white', linewidth=1)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', color='white', fontsize=9, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#aaaaaa', lw=1.5))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.15, label, ha='center', fontsize=7, color='#aaaaaa')

# Nodes
draw_node(ax, 7, 9, 'Portfolio Input', '#3355bb')
draw_node(ax, 7, 7.5, 'Supervisor Agent\n(LLM routing decision)', '#3355bb', width=3.5)

# Parallel agents
draw_node(ax, 2.5, 5.5, 'Market Risk\nAgent', '#f97316', width=2.2)
draw_node(ax, 5.5, 5.5, 'Credit Risk\nAgent', '#f97316', width=2.2)
draw_node(ax, 8.5, 5.5, 'Operational\nRisk Agent', '#f97316', width=2.2)
draw_node(ax, 11.5, 5.5, 'Macro\nAgent', '#f97316', width=2.2)

draw_node(ax, 7, 4, 'RAG Agent\n(Basel III citations)', '#7c3aed', width=3)
draw_node(ax, 7, 2.8, 'Aggregator\n(Composite risk score)', '#059669', width=3)

draw_node(ax, 4, 1.5, 'HITL Review\n(interrupt())', '#dc2626', width=2.5)
draw_node(ax, 10, 1.5, 'Report Agent\n(Final briefing)', '#059669', width=2.5)

# Arrows
draw_arrow(ax, 7, 8.6, 7, 7.9)
draw_arrow(ax, 7, 7.1, 2.5, 5.9, 'Send (parallel)')
draw_arrow(ax, 7, 7.1, 5.5, 5.9)
draw_arrow(ax, 7, 7.1, 8.5, 5.9)
draw_arrow(ax, 7, 7.1, 11.5, 5.9)

for x in [2.5, 5.5, 8.5, 11.5]:
    draw_arrow(ax, x, 5.1, 7, 4.3)

draw_arrow(ax, 7, 3.65, 7, 3.15)
draw_arrow(ax, 5.5, 2.65, 4.2, 1.8, 'HIGH/CRITICAL')
draw_arrow(ax, 8.5, 2.65, 9.8, 1.8, 'LOW/MEDIUM')
draw_arrow(ax, 5.3, 1.5, 9.7, 1.5, 'analyst approves')

# Labels
ax.text(7, 9.7, 'RiskLens — LangGraph Architecture', ha='center', fontsize=13, color='white', fontweight='bold')
ax.text(1.5, 6.2, 'Parallel via Send API', ha='center', fontsize=8, color='#aaaaaa', style='italic')

ax.set_facecolor('#0d0d18')
fig.patch.set_facecolor('#0d0d18')
plt.tight_layout()
plt.show()

print('Key architectural points:')
print('1. Supervisor → specialist agents: NOT sequential, parallel via Send API')
print('2. State flows through all nodes: each agent reads and adds to the shared state')
print('3. HITL interrupt: graph truly pauses, state persisted to SQLite')
print('4. Conditional edge: HIGH/CRITICAL → HITL, LOW/MEDIUM → direct to report')

---
## Summary

This notebook covered the key methodological foundations of RiskLens:

| Concept | Key Formula | Basel III Relevance |
|---|---|---|
| Historical VaR | percentile of returns distribution | Pillar 1 market risk capital |
| Parametric VaR | -(μ + z·σ) | Comparison benchmark |
| CVaR / ES | Average loss beyond VaR | **Basel IV FRTB primary metric** |
| Annualisation | σ_annual = σ_daily × √252 | All volatility calculations |
| Distance to Default | (E[Asset] - Debt) / (σ × √T) | Proxy credit risk PD |
| Yield Curve Inversion | 10Y-2Y spread < 0 | Macro risk / duration risk |
| HHI Concentration | Σ w_i² (normalised) | Portfolio diversification |

The LangGraph architecture allows these calculations to run as autonomous agents that share state, execute in parallel, and are supervised by an LLM that decides which analyses are needed — making the system adaptive rather than a fixed pipeline.